# 01 — Explore signals

Quick look at one record from Part_1: raw vs filtered traces, peak detection, BP labels. Use this to sanity-check preprocessing and to pick QC thresholds before running the full feature build.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
from bme_ml import SAMPLE_RATE_HZ
from bme_ml.paths import setup_paths
from bme_ml.data_loader import iter_records
from bme_ml.preprocessing import preprocess_record, window_record, is_valid_segment
from bme_ml.labels import label_segment

paths = setup_paths()

In [ ]:
# Grab the first record from Part_1.mat and plot 10 seconds of each channel.
part = paths.raw / 'Part_1.mat'
rec = next(iter_records(part))
print('record shape:', rec.shape)

rec_p = preprocess_record(rec)
n = SAMPLE_RATE_HZ * 10
t = np.arange(n) / SAMPLE_RATE_HZ

fig, axes = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t, rec_p[2, :n]); axes[0].set_ylabel('ECG (filtered)')
axes[1].plot(t, rec_p[0, :n]); axes[1].set_ylabel('PPG (filtered)')
axes[2].plot(t, rec_p[1, :n]); axes[2].set_ylabel('ABP (mmHg)')
axes[2].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

In [ ]:
# QC + label distribution across the first ~50 records.
from collections import Counter

qc = Counter()
labels = Counter()
sbp_dbp = []

for i, rec in enumerate(iter_records(part)):
    if i >= 50:
        break
    rec_p = preprocess_record(rec)
    for seg in window_record(rec_p):
        if not is_valid_segment(seg):
            qc['rejected'] += 1
            continue
        qc['accepted'] += 1
        lbl = label_segment(seg[1])
        if lbl is None:
            qc['no_peaks'] += 1
            continue
        labels[lbl.binary] += 1
        sbp_dbp.append((lbl.sbp, lbl.dbp))

print('QC :', dict(qc))
print('labels (0=Normal, 1=Abnormal):', dict(labels))

if sbp_dbp:
    s, d = zip(*sbp_dbp)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(d, s, s=8, alpha=0.5)
    ax.axhline(130, color='r', lw=0.5); ax.axvline(80, color='r', lw=0.5)
    ax.set_xlabel('DBP (mmHg)'); ax.set_ylabel('SBP (mmHg)')
    ax.set_title('Per-segment SBP vs DBP — red lines mark Normal threshold')
    plt.show()